# L50 Proof Replay

This notebook replays the bounded Communication pack proof on fixed repo state.

## Proof Focus

- One declared Communication authority
- One registry-backed Communication persona
- One deterministic pack-state rule
- One bounded supported-action total


## Fixed-State Replay

Re-run the authority and runtime checks against the same repo state.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import yaml

from smarthaus_common.department_pack import build_department_pack
from smarthaus_common.json_store import JsonStore

repo_root = Path.cwd()
authority_path = repo_root / 'registry' / 'department_pack_communication_v1.yaml'
payload = yaml.safe_load(authority_path.read_text(encoding='utf-8'))

assert payload['department']['id'] == 'communication'
assert payload['kpis']['required_personas'] == 1
assert payload['kpis']['supported_action_count'] == 7
assert list(payload['personas']) == ['outreach-coordinator']


In [ ]:
with TemporaryDirectory() as temp_dir:
    pack = build_department_pack('communication', store=JsonStore(Path(temp_dir)))

assert pack['department']['id'] == 'communication'
assert pack['summary']['persona_count'] == 1
assert pack['summary']['active_persona_count'] == 1
assert pack['summary']['supported_action_count'] == 7
assert pack['summary']['pack_state'] == 'ready'
assert [persona['persona_id'] for persona in pack['personas']] == ['outreach-coordinator']
assert pack['personas'][0]['approval_profile'] == 'high-impact'

assert pack['bounded_claims']['unsupported'][0] == 'finance or HR approval workflows'
